# Taller II · Evaluación Sumativa 3
**MCDI502 · Gestión de Datos y Tecnologías Big Data · Grupo 4**

Integrantes: Felipe Díaz Toro · Ninoska Yévenes Hernández · Guillermo Subiabre Chacón · Pablo Ríos Passteni

## Introducción

En este taller armamos, de punta a punta, un flujo de ingeniería de datos con Apache Spark. Tomamos datos de cuatro fuentes distintas, los dejamos en el mismo formato de trabajo (DataFrames de Spark), los limpiamos y transformamos, los integramos en una sola tabla y respondemos preguntas sobre ellos con Spark SQL.

Los datos reales casi nunca vienen de un solo lugar ni con la misma forma. Aquí combinamos dos fuentes de la web y dos de sensores:

| Fuente | Qué aporta | Cómo se accede | DataFrame |
|---|---|---|---|
| OpenMeteo | Clima exterior: temperatura, humedad, precipitación, viento e índice UV | API web (JSON) | `df_openmeteo` |
| AQICN | Calidad del aire: PM2.5, PM10, O3, NO2 y SO2 | API web (JSON) con token | `df_aqicn` |
| MongoDB (ESP32) | Altura en pies, presión atmosférica y monóxido de carbono | Base de datos no relacional | `df_mongo_bpm` |
| MySQL (ESP32) | Temperatura interior y humedad | Base de datos relacional, vía API en PHP | `df_mysql_dht` |

**Cómo avanzamos.**
1. **Configuración del entorno (sección 1):** Spark, variables de entorno, `SparkSession`, `SparkContext` y conectores.
2. **Ingesta (sección 2):** conectamos cada fuente y dejamos exactamente 100 registros en su DataFrame.
3. **Preprocesamiento (sección 3):** creamos dos columnas calculadas: `altura_metros` (conversión de pies a metros) y `condicion_clima` (clasificación según temperatura y humedad).
4. **Integración (sección 4):** unimos las cuatro fuentes en `df_unificado`. Como no comparten una clave natural, justificamos la estrategia elegida.
5. **Consultas (sección 5):** creamos una tabla temporal por DataFrame y respondemos las siete consultas del enunciado (C1 a C7) con Spark SQL.
6. **Notas para el video (sección 6):** resumen de las decisiones técnicas que explicamos al grabar.

# 1. Configuración inicial del entorno
El notebook se ejecuta en VS Code con el entorno de conda del grupo (`grupo4_mcdi502`), que ya trae Java 17 y PySpark. Se crean las variables de entorno, la `SparkSession` y el `SparkContext`, y se instalan las librerías para las fuentes de datos (requests, pandas, pymongo, mysql-connector-python). El conector de Spark para MongoDB se descarga desde Maven Central la primera vez que se ejecuta la celda 1.3 y queda en la carpeta `jars` de la instalación de Spark del entorno.

La configuración funciona igual en macOS y en Windows: la ruta de Java se detecta según el sistema y el conector se carga sin necesidad de instalar Hadoop ni `winutils.exe`.

In [1]:
# 1.1 Librerías para las fuentes de datos (si ya están instaladas, pip no hace nada)
# setuptools: restaura distutils, que PySpark 3.5.x necesita para createDataFrame(pandas_df) y que Python 3.12 quitó de la librería estándar
# pandas<3: PySpark 3.5 está hecho para pandas 2.x; además, en Windows el control de aplicaciones puede bloquear las DLL de pandas 3
%pip install -q requests "pandas<3" pymongo mysql-connector-python setuptools

Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1.2 Variables de entorno: JAVA_HOME apunta al Java del entorno de conda
# conda instala Java en Library/lib/jvm en Windows y en lib/jvm en macOS/Linux
import os, sys
candidatos = [os.path.join(sys.prefix, "Library", "lib", "jvm"),   # Windows
              os.path.join(sys.prefix, "lib", "jvm")]               # macOS / Linux
java_home = next((p for p in candidatos if os.path.exists(p)), None)
if java_home is None:
    raise FileNotFoundError("No se encontró Java en el entorno de conda: instalar openjdk 17 en grupo4_mcdi502")
os.environ["JAVA_HOME"] = java_home

# Raíz del repo: el kernel puede partir en notebooks/ (lo habitual en VS Code) o en la raíz
RAIZ = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("Raíz del repo =", RAIZ)

JAVA_HOME = C:\Users\pablo\anaconda3\envs\grupo4_mcdi502\Library\lib\jvm
Raíz del repo = C:\Users\pablo\Desktop\MCDI502 Gestión de Datos y Tecnologías Big Data\proyecto-grupo4-mcdi502


In [3]:
# 1.3 SparkSession y SparkContext
# Conector de MongoDB para Spark (versión 3.0.1, la misma del notebook base del docente) y sus dependencias.
# Se descargan una sola vez desde Maven Central a la carpeta jars/ de la instalación de Spark del entorno,
# que Spark carga automáticamente al iniciar. Así funciona igual en macOS y en Windows:
#  - spark.jars.packages y spark.jars exigen winutils.exe (HADOOP_HOME) en Windows
#  - spark.driver.extraClassPath falla en Windows si la ruta del proyecto tiene espacios o tildes
import requests, findspark
findspark.init()
from pyspark.sql import SparkSession, functions as F

MAVEN = "https://repo1.maven.org/maven2/org/mongodb/"
JARS = {"mongo-spark-connector_2.12-3.0.1.jar": "spark/mongo-spark-connector_2.12/3.0.1/",
        "mongodb-driver-sync-4.0.5.jar": "mongodb-driver-sync/4.0.5/",
        "mongodb-driver-core-4.0.5.jar": "mongodb-driver-core/4.0.5/",
        "bson-4.0.5.jar": "bson/4.0.5/"}
CARPETA_JARS = os.path.join(os.environ["SPARK_HOME"], "jars")   # findspark.init() define SPARK_HOME
for nombre, ruta in JARS.items():
    destino = os.path.join(CARPETA_JARS, nombre)
    if not os.path.exists(destino):
        respuesta = requests.get(MAVEN + ruta + nombre, timeout=60)
        respuesta.raise_for_status()
        with open(destino, "wb") as archivo:
            archivo.write(respuesta.content)
print("Conector de MongoDB en", CARPETA_JARS)

# Spark lee la carpeta jars/ al arrancar la JVM: si ya había una sesión abierta antes de la descarga, reiniciar el kernel
spark = (SparkSession.builder
         .master("local[*]")
         .appName("Taller2 Grupo4")
         .getOrCreate())
sc = spark.sparkContext
print("Spark", spark.version, "| aplicación:", sc.appName)

Conector de MongoDB en C:\Users\pablo\anaconda3\envs\grupo4_mcdi502\Lib\site-packages\pyspark\jars


Spark 3.5.9 | aplicación: Taller2 Grupo4


In [4]:
# 1.4 Configuración de fuentes. Los secretos van en un archivo .env en la raíz del repo (está en .gitignore)
# Formato del .env, una línea por valor, sin comillas:
#   AQICN_TOKEN=...
#   MONGO_URI=...
def cargar_env(ruta_env=os.path.join(RAIZ, ".env")):
    if not os.path.exists(ruta_env):
        raise FileNotFoundError(f"No se encontró {ruta_env}: crearlo a partir de .env.example")
    valores = {}
    with open(ruta_env, encoding="utf-8") as archivo:
        for linea in archivo:
            if "=" in linea and not linea.strip().startswith("#"):
                k, v = linea.strip().split("=", 1)
                valores[k.strip()] = v.strip()
    return valores

_env = cargar_env()
faltan = [k for k in ("AQICN_TOKEN", "MONGO_URI")
          if not _env.get(k) or _env[k].startswith(("PEGAR", "tu_"))]
if faltan:
    print("⚠ Completar en .env:", ", ".join(faltan))

AQICN_TOKEN = _env.get("AQICN_TOKEN", "PEGAR_TOKEN_PROPIO")   # token gratuito de https://aqicn.org/data-platform/token/
AQICN_CIUDAD = "santiago"                                      # ciudad/región del grupo
MONGO_URI = _env.get("MONGO_URI", "PEGAR_URI_DEL_DOCENTE")     # URI del anuncio del docente (Taller II); solo en el .env
MONGO_DB, MONGO_COL = "telemetria", "lecturas"                 # base y colección indicadas por el docente para el Taller II
MYSQL_URL = "http://mgtt.comunidadingenieria.cl/obtener_datos.php"
N_REGISTROS = 100                                              # el enunciado pide 100 por DataFrame

# 2. Ingesta de datos y creación de DataFrames (ID3.1)
Cada fuente se consulta, se deja en exactamente 100 registros y se convierte a un DataFrame de Spark.

## 2.1 OpenMeteo → `df_openmeteo`  *(Ninoska)*
Se pide una serie horaria de los últimos días, que entrega más de 100 registros de una sola vez y con fecha, lo que sirve para las consultas C2 y C3. Las coordenadas son las del notebook base (Viña del Mar).

In [5]:
import requests, pandas as pd

LAT, LON = -33.02457, -71.55183
VARS = ["temperature_2m", "relative_humidity_2m", "precipitation",
        "wind_speed_10m", "wind_direction_10m", "uv_index"]
url = ("https://api.open-meteo.com/v1/forecast"
       f"?latitude={LAT}&longitude={LON}&hourly={','.join(VARS)}"
       "&past_days=7&forecast_days=1&timezone=America/Santiago")
h = requests.get(url, timeout=30).json()["hourly"]
pdf_om = pd.DataFrame(h).rename(columns={"time": "tiempo"})
pdf_om["tiempo"] = pd.to_datetime(pdf_om["tiempo"])
# últimos 100 registros que ya ocurrieron (no pronóstico).
# Se compara en hora de Santiago (no en la hora del computador que ejecute el notebook):
# así el corte entre pasado y pronóstico es el mismo sin importar dónde se corra.
ahora_santiago = pd.Timestamp.now(tz="America/Santiago").tz_localize(None)
pdf_om = pdf_om[pdf_om["tiempo"] <= ahora_santiago].tail(N_REGISTROS).reset_index(drop=True)

df_openmeteo = spark.createDataFrame(pdf_om)
print("Registros:", df_openmeteo.count())
df_openmeteo.show(5, truncate=False)

Registros: 100


+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+
|2026-11-06 21:00:00|14.3          |63                  |0.0          |11.7          |180               |0.0     |
|2026-11-06 22:00:00|13.5          |67                  |0.0          |12.1          |176               |0.0     |
|2026-11-06 23:00:00|13.4          |69                  |0.0          |13.9          |182               |0.0     |
|2026-11-07 00:00:00|13.3          |70                  |0.0          |14.0          |181               |0.0     |
|2026-11-07 01:00:00|13.1          |71                  |0.0          |14.9          |179               |0.0     |
+-------------------+--------------+--------------------+-------------+---------

## 2.2 AQICN → `df_aqicn`  *(Guillermo)*
El token `demo` del notebook base no corresponde a la región del grupo; se usa el token propio de la celda 1.4. Cada consulta devuelve la última medición publicada por la estación, que se actualiza cada hora, por lo que las 100 filas pueden repetir valores. La columna `tiempo_captura` registra el momento de cada consulta y permite distinguirlas y ordenarlas.

In [6]:
import time
rows = []
url = f"http://api.waqi.info/feed/{AQICN_CIUDAD}/?token={AQICN_TOKEN}"
while len(rows) < N_REGISTROS:
    d = requests.get(url, timeout=30).json()
    if d.get("status") != "ok":
        # se detiene con un mensaje claro (antes un token inválido dejaba 0 filas y fallaba recién en createDataFrame)
        raise RuntimeError(f"AQICN respondió {d}. Revisar AQICN_TOKEN en el .env")
    iaqi = d["data"]["iaqi"]
    rows.append({
        "tiempo_captura": pd.Timestamp.now(),   # momento de la consulta
        "tiempo": d["data"]["time"]["s"],       # hora de la medición que informa la estación
        "PM2_5": float(iaqi.get("pm25", {}).get("v", "nan")),
        "PM10": float(iaqi.get("pm10", {}).get("v", "nan")),
        "O3":   float(iaqi.get("o3",   {}).get("v", "nan")),
        "NO2":  float(iaqi.get("no2",  {}).get("v", "nan")),
        "SO2":  float(iaqi.get("so2",  {}).get("v", "nan")),
    })
    time.sleep(1)

df_aqicn = spark.createDataFrame(pd.DataFrame(rows))
print("Registros:", df_aqicn.count())
df_aqicn.show(5, truncate=False)

Registros: 100


+--------------------------+-------------------+-----+----+----+----+---+
|tiempo_captura            |tiempo             |PM2_5|PM10|O3  |NO2 |SO2|
+--------------------------+-------------------+-----+----+----+----+---+
|2026-11-11 00:52:28.799096|2026-11-09 20:00:00|25.0 |29.0|18.4|12.7|NaN|
|2026-11-11 00:52:30.500275|2026-11-09 20:00:00|25.0 |29.0|18.4|12.7|NaN|
|2026-11-11 00:52:32.160094|2026-11-09 20:00:00|25.0 |29.0|18.4|12.7|NaN|
|2026-11-11 00:52:33.821211|2026-11-09 20:00:00|25.0 |29.0|18.4|12.7|NaN|
|2026-11-11 00:52:35.485582|2026-11-09 20:00:00|25.0 |29.0|18.4|12.7|NaN|
+--------------------------+-------------------+-----+----+----+----+---+
only showing top 5 rows



## 2.3 MongoDB → `df_mongo_bpm`  *(Pablo)*
Se lee la colección del docente con el conector de Spark. Las variables a capturar son la altura en pies (`altitudFt`), la presión atmosférica (`presionAtm`) y el monóxido de carbono (`ppmCO`), más `fecha` para ordenar. La colección tiene decenas de miles de filas, por lo que se ordena por `fecha` descendente y se toman las últimas 100.

Después se dejan en orden ascendente, igual que las otras fuentes, y se congelan con `localCheckpoint()`. Spark vuelve a consultar MongoDB en cada acción y la colección recibe una lectura nueva cada ~17 segundos, así que sin congelarlas las 100 filas cambiarían de una celda a otra.

In [7]:
df_mongo_raw = (spark.read.format("com.mongodb.spark.sql.DefaultSource")
                .option("uri", MONGO_URI)
                .option("database", MONGO_DB)
                .option("collection", MONGO_COL)
                .load())
df_mongo_raw.printSchema()

root
 |-- __v: integer (nullable = true)
 |-- _id: struct (nullable = true)
 |    |-- oid: string (nullable = true)
 |-- altitudFt: double (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- ppmCO: double (nullable = true)
 |-- presionAtm: double (nullable = true)
 |-- presionPa: integer (nullable = true)
 |-- temperatura: double (nullable = true)



In [8]:
from pyspark.sql import functions as F
# Columnas reales de la colección (ver printSchema arriba)
cols_mongo = ["fecha", "altitudFt", "presionAtm", "ppmCO"]
df_mongo_bpm = (df_mongo_raw.select(*cols_mongo)
                .orderBy(F.col("fecha").desc())
                .limit(N_REGISTROS)      # las 100 lecturas más recientes
                .orderBy("fecha")        # luego en orden ascendente, como las otras fuentes
                .localCheckpoint())      # congela las 100 filas (la colección sigue recibiendo datos)
print("Registros:", df_mongo_bpm.count())
df_mongo_bpm.show(5, truncate=False)

Registros: 100
+-----------------------+---------+----------+-----+
|fecha                  |altitudFt|presionAtm|ppmCO|
+-----------------------+---------+----------+-----+
|2026-11-11 00:27:09.681|802.32   |0.97135   |3.87 |
|2026-11-11 00:27:26.798|797.85   |0.97132   |3.87 |
|2026-11-11 00:27:43.906|802.32   |0.971359  |3.87 |
|2026-11-11 00:28:01.017|802.88   |0.97133   |3.87 |
|2026-11-11 00:28:18.305|804.56   |0.97134   |3.87 |
+-----------------------+---------+----------+-----+
only showing top 5 rows



## 2.4 MySQL → `df_mysql_dht`  *(Pablo)*
La base MySQL se consulta a través de la API PHP, que devuelve JSON con `id`, `temperatura`, `humedad` y `fecha`. La API entrega las últimas 1.000 lecturas ordenadas de la más nueva a la más antigua, por lo que se ordena por `fecha` antes de tomar las 100 más recientes. La humedad se renombra `humedad_interior` para distinguirla de la humedad exterior de OpenMeteo.

In [9]:
data = requests.get(MYSQL_URL, timeout=30).json()
pdf_my = pd.DataFrame(data)
pdf_my["fecha"] = pd.to_datetime(pdf_my["fecha"])
pdf_my["temperatura_interior"] = pdf_my["temperatura"].astype(float)
pdf_my["humedad_interior"] = pdf_my["humedad"].astype(float)
# la API entrega de la más nueva a la más antigua: se ordena antes de recortar para quedarse con las 100 más recientes
pdf_my = (pdf_my.sort_values("fecha").tail(N_REGISTROS)
          [["id", "fecha", "temperatura_interior", "humedad_interior"]]
          .reset_index(drop=True))

df_mysql_dht = spark.createDataFrame(pdf_my)
print("Registros:", df_mysql_dht.count())
df_mysql_dht.show(5, truncate=False)

Registros: 100


+------+-------------------+--------------------+----------------+
|id    |fecha              |temperatura_interior|humedad_interior|
+------+-------------------+--------------------+----------------+
|368092|2026-11-10 19:27:26|23.4                |57.8            |
|368093|2026-11-10 19:27:43|23.4                |57.7            |
|368094|2026-11-10 19:28:00|23.4                |57.8            |
|368095|2026-11-10 19:28:18|23.4                |57.8            |
|368096|2026-11-10 19:28:35|23.4                |57.7            |
+------+-------------------+--------------------+----------------+
only showing top 5 rows



# 3. Preprocesamiento y transformación de datos (RA4, ID4.2)

## 3.1 Columna `altura_metros` en `df_mongo_bpm`  *(Pablo)*
1 pie = 0,3048 metros.

In [10]:
from pyspark.sql import functions as F
df_mongo_bpm = df_mongo_bpm.withColumn("altura_metros", F.round(F.col("altitudFt") * 0.3048, 4))
df_mongo_bpm.select("altitudFt", "altura_metros").show(5)

+---------+-------------+
|altitudFt|altura_metros|
+---------+-------------+
|   802.32|     244.5471|
|   797.85|     243.1847|
|   802.32|     244.5471|
|   802.88|     244.7178|
|   804.56|     245.2299|
+---------+-------------+
only showing top 5 rows



## 3.2 Columna `condicion_clima` en `df_openmeteo`  *(Ninoska)*
Lógica del enunciado: calor y humedad (T > 30 y H > 70), calor seco (T > 30 y H ≤ 70), clima moderado (T ≤ 30) y, en otro caso (datos nulos), condición desconocida.

In [11]:
t, hum = F.col("temperature_2m"), F.col("relative_humidity_2m")
df_openmeteo = df_openmeteo.withColumn(
    "condicion_clima",
    F.when((t > 30) & (hum > 70), "CALOR Y HUMEDAD")
     .when((t > 30) & (hum <= 70), "CALOR SECO")
     .when(t <= 30, "CLIMA MODERADO")
     .otherwise("CONDICION DESCONOCIDA"))
df_openmeteo.groupBy("condicion_clima").count().show()

+---------------+-----+
|condicion_clima|count|
+---------------+-----+
| CLIMA MODERADO|   97|
|     CALOR SECO|    3|
+---------------+-----+



# 4. Integración de datos (RA3)  *(Felipe)*
**Estrategia:** las cuatro fuentes son heterogéneas y no comparten una clave natural (cada una tiene su propia marca de tiempo y su propio origen). Cada fuente se ordena por su propia marca de tiempo, de la más antigua a la más reciente, se numeran sus filas (`idx`) y se unen **por índice de fila**. Es una decisión simplificadora y queda documentada: no se afirma una correspondencia temporal real entre sensores y APIs. Antes de unir se verifica que las cuatro tablas tengan exactamente 100 registros, para no perder filas en el join, y se conserva la marca de tiempo de cada fuente para que el cruce sea transparente.

**Observación:** MongoDB y MySQL reciben datos del mismo ESP32. Sus fechas coinciden al segundo con un desfase fijo de 5 horas entre ambas bases, por lo que entre esas dos fuentes la fila *n* corresponde aproximadamente a la misma lectura.

In [12]:
from pyspark.sql.window import Window

def con_indice(df, col_orden):
    # numera las filas según la marca de tiempo de cada fuente (1 = la más antigua)
    return df.withColumn("idx", F.row_number().over(Window.orderBy(F.col(col_orden))))

for nombre, df in {"openmeteo": df_openmeteo, "aqicn": df_aqicn,
                   "mongo": df_mongo_bpm, "mysql": df_mysql_dht}.items():
    n = df.count()
    assert n == N_REGISTROS, f"{nombre} tiene {n} filas, se esperaban {N_REGISTROS}"

om = con_indice(df_openmeteo, "tiempo").select(
    "idx", F.col("tiempo").alias("tiempo_openmeteo"), "temperature_2m", "relative_humidity_2m",
    "precipitation", "wind_speed_10m", "wind_direction_10m", "uv_index", "condicion_clima")
aq = con_indice(df_aqicn, "tiempo_captura").select(
    "idx", F.col("tiempo").alias("tiempo_aqicn"), "PM2_5", "PM10", "O3", "NO2", "SO2")
mo = con_indice(df_mongo_bpm, "fecha").select(
    "idx", F.col("fecha").alias("fecha_mongo"), "altitudFt", "altura_metros", "presionAtm", "ppmCO")
my = con_indice(df_mysql_dht, "fecha").select(
    "idx", F.col("fecha").alias("fecha_mysql"), "temperatura_interior", "humedad_interior")

df_unificado = (om.join(aq, "idx").join(mo, "idx").join(my, "idx")).orderBy("idx")
print("Filas:", df_unificado.count(), "| Columnas:", len(df_unificado.columns))
df_unificado.show(5, truncate=False)

Filas: 100 | Columnas: 23


+---+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+-------------------+-----+----+----+----+---+-----------------------+---------+-------------+----------+-----+-------------------+--------------------+----------------+
|idx|tiempo_openmeteo   |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|tiempo_aqicn       |PM2_5|PM10|O3  |NO2 |SO2|fecha_mongo            |altitudFt|altura_metros|presionAtm|ppmCO|fecha_mysql        |temperatura_interior|humedad_interior|
+---+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+-------------------+-----+----+----+----+---+-----------------------+---------+-------------+----------+-----+-------------------+--------------------+----------------+
|1  |2026-11-06 21:00:00|14.3          |63                  |0.0          |11.7          |1

# 5. Consultas y extracción de datos (ID3.1, ID4.1)
Primero se crea una tabla temporal por DataFrame con Spark SQL.

In [13]:
df_openmeteo.createOrReplaceTempView("openmeteo")
df_aqicn.createOrReplaceTempView("aqicn")
df_mongo_bpm.createOrReplaceTempView("mongo_bpm")
df_mysql_dht.createOrReplaceTempView("mysql_dht")
# spark.catalog.listTables() y SHOW TABLES fallan en Windows sin winutils.exe; tableExists funciona en ambos sistemas
for vista in ["openmeteo", "aqicn", "mongo_bpm", "mysql_dht"]:
    print(f"{vista}: {'creada' if spark.catalog.tableExists(vista) else 'NO creada'}")

openmeteo: creada
aqicn: creada
mongo_bpm: creada
mysql_dht: creada


**C1** · 10 registros con mayor temperatura  *(Ninoska)*

In [14]:
spark.sql("SELECT * FROM openmeteo ORDER BY temperature_2m DESC LIMIT 10").show(truncate=False)

+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|2026-11-08 14:00:00|30.4          |43                  |0.0          |8.7           |312               |8.9     |CALOR SECO     |
|2026-11-08 15:00:00|30.4          |40                  |0.0          |9.3           |313               |8.55    |CALOR SECO     |
|2026-11-08 13:00:00|30.2          |42                  |0.0          |7.8           |322               |8.55    |CALOR SECO     |
|2026-11-08 12:00:00|29.6          |40                  |0.0          |6.4           |324               |7.45    |CLIMA MODERADO |
|2026-11-08 16:00:00|28.6          |44                  |0.0          |10.2        

**C2** · Promedio de humedad relativa para un mes y día específicos  *(Ninoska)*. Se muestran los días disponibles y se elige el último día completo (24 horas), así la consulta sigue siendo válida aunque la ventana de 100 horas avance.

In [15]:
dias = spark.sql("""SELECT month(tiempo) AS mes, day(tiempo) AS dia, COUNT(*) AS horas
                    FROM openmeteo GROUP BY 1, 2 ORDER BY 1, 2""")
dias.show(20)
ultimo_dia = dias.filter("horas = 24").orderBy(F.desc("mes"), F.desc("dia")).first()
MES, DIA = ultimo_dia["mes"], ultimo_dia["dia"]
spark.sql(f"""SELECT {MES} AS mes, {DIA} AS dia, ROUND(AVG(relative_humidity_2m), 2) AS humedad_promedio
              FROM openmeteo WHERE month(tiempo) = {MES} AND day(tiempo) = {DIA}""").show()

+---+---+-----+
|mes|dia|horas|
+---+---+-----+
| 11|  6|    3|
| 11|  7|   24|
| 11|  8|   24|
| 11|  9|   24|
| 11| 10|   24|
| 11| 11|    1|
+---+---+-----+



+---+---+----------------+
|mes|dia|humedad_promedio|
+---+---+----------------+
| 11| 10|           75.83|
+---+---+----------------+



**C3** · Registros con precipitación > 2 mm durante la última semana  *(Ninoska)*. La referencia es la hora más reciente de los datos; los 100 registros horarios cubren unos 4 días.

In [16]:
spark.sql("""SELECT COUNT(*) AS registros_sobre_2mm FROM openmeteo
              WHERE precipitation > 2
                AND tiempo >= (SELECT max(tiempo) FROM openmeteo) - INTERVAL 7 DAYS""").show()

+-------------------+
|registros_sobre_2mm|
+-------------------+
|                  0|
+-------------------+



**C4** · Promedio de PM2.5, PM10, O3, NO2 y SO2  *(Guillermo)*

In [17]:
spark.sql("SELECT AVG(PM2_5) AS pm25, AVG(PM10) AS pm10, AVG(O3) AS o3, AVG(NO2) AS no2, AVG(SO2) AS so2 FROM aqicn").show()

+----+----+------------------+------------------+---+
|pm25|pm10|                o3|               no2|so2|
+----+----+------------------+------------------+---+
|25.0|29.0|18.400000000000006|12.699999999999998|NaN|
+----+----+------------------+------------------+---+



**C5** · Registros con NO2 superior a 4  *(Guillermo)*

In [18]:
spark.sql("SELECT COUNT(*) AS registros_no2_mayor_4 FROM aqicn WHERE NO2 > 4").show()

+---------------------+
|registros_no2_mayor_4|
+---------------------+
|                  100|
+---------------------+



**C6** · Primeros 10 registros de `df_mongo_bpm` con `altura_metros`  *(Pablo)*

In [19]:
spark.sql("SELECT * FROM mongo_bpm ORDER BY fecha LIMIT 10").show(truncate=False)

+-----------------------+---------+----------+-----+-------------+
|fecha                  |altitudFt|presionAtm|ppmCO|altura_metros|
+-----------------------+---------+----------+-----+-------------+
|2026-11-11 00:27:09.681|802.32   |0.97135   |3.87 |244.5471     |
|2026-11-11 00:27:26.798|797.85   |0.97132   |3.87 |243.1847     |
|2026-11-11 00:27:43.906|802.32   |0.971359  |3.87 |244.5471     |
|2026-11-11 00:28:01.017|802.88   |0.97133   |3.87 |244.7178     |
|2026-11-11 00:28:18.305|804.56   |0.97134   |3.87 |245.2299     |
|2026-11-11 00:28:35.41 |803.44   |0.971261  |3.87 |244.8885     |
|2026-11-11 00:28:52.51 |803.16   |0.97132   |3.87 |244.8032     |
|2026-11-11 00:29:09.626|804.0    |0.97134   |3.84 |245.0592     |
|2026-11-11 00:29:26.92 |805.68   |0.971281  |3.87 |245.5713     |
|2026-11-11 00:29:44.075|803.44   |0.971251  |3.87 |244.8885     |
+-----------------------+---------+----------+-----+-------------+



**C7** · Primeros 10 registros de `df_openmeteo` con `condicion_clima`  *(Ninoska)*

In [20]:
spark.sql("SELECT * FROM openmeteo LIMIT 10").show(truncate=False)

+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|tiempo             |temperature_2m|relative_humidity_2m|precipitation|wind_speed_10m|wind_direction_10m|uv_index|condicion_clima|
+-------------------+--------------+--------------------+-------------+--------------+------------------+--------+---------------+
|2026-11-06 21:00:00|14.3          |63                  |0.0          |11.7          |180               |0.0     |CLIMA MODERADO |
|2026-11-06 22:00:00|13.5          |67                  |0.0          |12.1          |176               |0.0     |CLIMA MODERADO |
|2026-11-06 23:00:00|13.4          |69                  |0.0          |13.9          |182               |0.0     |CLIMA MODERADO |
|2026-11-07 00:00:00|13.3          |70                  |0.0          |14.0          |181               |0.0     |CLIMA MODERADO |
|2026-11-07 01:00:00|13.1          |71                  |0.0          |14.9        

# 6. Notas para el video
- Qué conexión usó cada fuente y por qué: OpenMeteo (serie horaria para llegar a 100 registros con fecha), AQICN (token propio), MongoDB (conector de Spark), MySQL (API PHP).
- Cómo se configuró el entorno para que funcione en macOS y en Windows (ruta de Java y conector de MongoDB en la carpeta `jars` de Spark).
- Por qué MongoDB se congela con `localCheckpoint()` y por qué MySQL se ordena antes de recortar.
- Por qué la integración es por índice y qué limitación tiene.
- Qué resultado dio cada consulta (C1 a C7) y qué significa.